In [1]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

# Same project layout as the previous notebooks.
project_root = Path.cwd().parent

notebook_dir = Path().resolve()

data_folder = project_root / "data" / "resources"
Pilot = data_folder / "PilotStudy_All"

# Get graph prevouly fitered
GRAPH_PATH = data_folder / "generated" / "modified_graph.gml"
# Get the original graph for matching ids
human1_xml = data_folder / "Human-GEM.xml"
graph = nx.read_gml(GRAPH_PATH)

if graph.is_directed() or graph.is_multigraph():
    raise ValueError("Use the simple, undirected reference graph.")

if nx.number_of_selfloops(graph) > 0:
    raise ValueError("The reference graph contains self-loops.")

if not nx.is_connected(graph):
    raise ValueError("This analysis requires the connected reference graph.")

graph_nodes = list(graph.nodes())
node_to_index = {
    node: index for index, node in enumerate(graph_nodes)
}
n = len(graph_nodes)

# Sampling settings.
K_VALUES = list(range(60, 201, 20))
N_RANDOM_PER_K = 2
BASE_SEED = 42

# Parameter settings for the later sensitivity analysis.
REFERENCE_D = 7.0
D_REF_VALUES = np.array([6.0, 6.5, 7.0, 7.5, 8.0])

rho = nx.density(graph)
mu = 1.0 - rho

# Optional: mapped graph-node IDs from an actual dataset.
# Leave as None to run entirely with generated configurations.
REFERENCE_NODE_IDS = None

if n < max(K_VALUES) + 3:
    raise ValueError(
        "The graph needs at least max(k) + 3 nodes "
        "to allow the planned three-node swaps."
    )

# Shortest-path distances on the full reference graph.
distance_matrix = np.full((n, n), np.inf, dtype=float)

for source, lengths in nx.all_pairs_shortest_path_length(graph):
    columns = np.fromiter(
        (node_to_index[node] for node in lengths),
        dtype=int,
    )
    distance_matrix[node_to_index[source], columns] = np.fromiter(
        lengths.values(), dtype=float
    )

print(f"Graph: {n} nodes, {graph.number_of_edges()} edges")
print(f"Density: {rho:.6f}")
print(f"k values: {K_VALUES}")
print(f"Distance matrix: {distance_matrix.shape}")

Graph: 3240 nodes, 7254 edges
Density: 0.001382
k values: [60, 80, 100, 120, 140, 160, 180, 200]
Distance matrix: (3240, 3240)
